# Data Lakehouse Tutorial - Part 1: Setup and Architecture

## Overview
This tutorial series demonstrates a complete modern data architecture using:
- **OLTP Database**: PostgreSQL with logical replication
- **Change Data Capture (CDC)**: Debezium + Kafka
- **Data Processing**: Apache Spark + Delta Lake
- **Object Storage**: MinIO (S3-compatible)
- **Orchestration**: Apache Airflow
- **Interactive Analysis**: Jupyter Lab

## Architecture Components

```
OLTP (PostgreSQL) 
       ↓ (CDC)
   Debezium + Kafka 
       ↓ (Batch Ingestion)
   Bronze Layer (Raw Data) 
       ↓ (Transformation)
   Silver Layer (Clean Data) 
       ↓ (Aggregation)
   Gold Layer (Business Views)
```

### Data Layers Explained

- **Bronze Layer**: Raw data from source systems, minimal transformations
- **Silver Layer**: Cleaned, validated, and deduplicated data
- **Gold Layer**: Business-ready aggregated data and metrics


## Environment Setup

### 1. Check Spark Configuration

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import json
from datetime import datetime, timedelta

print("Environment Variables:")
print(f"SPARK_MASTER: {os.environ.get('SPARK_MASTER', 'Not set')}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH', 'Not set')}")

### 2. Initialize Spark Session with Delta Lake and MinIO

In [ ]:
def create_spark_session():
    """
    Create Spark session configured for our data lakehouse environment.
    """
    spark = SparkSession.builder \
        .appName("DataLakehouseTutorial") \
        .config("spark.master", "spark://spark-master:7077") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    return spark

# Create Spark session
spark = create_spark_session()

print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")
print(f"Application Name: {spark.sparkContext.appName}")

### 3. Test MinIO Connectivity

In [ ]:
# Test MinIO connectivity by listing buckets
try:
    from minio import Minio
    
    minio_client = Minio(
        endpoint="minio:9000",
        access_key="minioadmin",
        secret_key="minioadmin123",
        secure=False
    )
    
    buckets = list(minio_client.list_buckets())
    print("MinIO Buckets:")
    for bucket in buckets:
        print(f"  - {bucket.name} (created: {bucket.creation_date})")
        
        # List some objects in each bucket (first 5)
        objects = list(minio_client.list_objects(bucket.name, recursive=True))
        if objects:
            print(f"    Objects in {bucket.name} (showing first 5):")
            for obj in objects[:5]:
                print(f"      - {obj.object_name} ({obj.size} bytes)")
        else:
            print(f"    {bucket.name} is empty")
    
except Exception as e:
    print(f"Error connecting to MinIO: {e}")

### 4. Service Status Check

In [ ]:
import requests
from urllib.parse import urljoin

def check_service_health():
    """
    Check the health of various services in our data pipeline.
    """
    services = {
        "Spark Master": "http://spark-master:8080",
        "MinIO API": "http://minio:9000/minio/health/live",
        "Kafka UI": "http://kafka-ui:8080",
        "Debezium UI": "http://debezium-ui:8080",
        "Airflow API": "http://airflow-apiserver:8080/health"
    }
    
    print("Service Health Check:")
    print("-" * 40)
    
    for service_name, url in services.items():
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                status = "✅ HEALTHY"
            else:
                status = f"⚠️  HTTP {response.status_code}"
        except requests.exceptions.RequestException as e:
            status = f"❌ UNAVAILABLE ({str(e)[:30]}...)"
        
        print(f"{service_name:15} {status}")

check_service_health()

## Data Pipeline Overview

### Current DAGs in Airflow

1. **init_and_seed_oltp**: Initialize database schema and create sample data
2. **stream_new_orders**: Generate streaming order data for CDC
3. **kafka_to_bronze**: Batch ingestion from Kafka to Bronze layer
4. **bronze_to_silver**: Data transformation and cleaning
5. **data_quality_checks**: Comprehensive quality monitoring

### Understanding the Data Flow

Let's examine what data looks like at each stage:

In [ ]:
def explore_layer_data(layer_name, table_name, limit=5):
    """
    Explore data in a specific layer and table.
    """
    try:
        path = f"s3a://{layer_name}/{table_name}/"
        
        if layer_name == "silver":
            df = spark.read.format("delta").load(path)
        else:
            df = spark.read.format("parquet").load(path)
        
        print(f"\n=== {layer_name.upper()} Layer - {table_name} ===")
        print(f"Total Records: {df.count()}")
        print(f"Schema:")
        df.printSchema()
        
        print(f"\nSample Data (first {limit} records):")
        df.show(limit, truncate=False)
        
        return df
        
    except Exception as e:
        print(f"No data found in {layer_name}/{table_name}: {e}")
        return None

# Try to explore existing data
tables_to_check = ["orders", "customers", "products"]
layers_to_check = ["bronze", "silver"]

for layer in layers_to_check:
    for table in tables_to_check:
        explore_layer_data(layer, table, limit=3)

## Next Steps

If you don't see data yet, that's expected! The next notebooks will guide you through:

1. **Part 2**: Running the initial data setup and CDC configuration
2. **Part 3**: Exploring Bronze layer data ingestion
3. **Part 4**: Understanding Silver layer transformations
4. **Part 5**: Building Gold layer aggregations
5. **Part 6**: Advanced analytics and monitoring

### Key Learning Objectives

By the end of this tutorial series, you will understand:
- How to build a scalable data lakehouse architecture
- Real-time data ingestion with CDC and Kafka
- Data transformation patterns with Spark and Delta Lake
- Data quality monitoring and validation
- Best practices for data pipeline orchestration

In [ ]:
# Clean up
spark.stop()
print("Spark session stopped.")